In [1]:
W = 10  # roll width
sizes = [2, 3, 5, 7]  # item sizes
demands = [61, 40, 32, 100]  # demand per item type

# Column Generation

In [2]:
from cutting_stock_col_gen import CuttingStockColGen

cs_cg = CuttingStockColGen(sizes, demands, wood_length=W)
solver, patterns, q_pattern, duals = cs_cg.master()
print("Best objective:", solver.Objective().Value())
print(f"demands = {demands}")
print(f"size, dual = {[(s, d) for s, d in zip(sizes, duals, strict=False)]}")
[(p, q_pattern[i].solution_value()) for i, p in enumerate(patterns)]

Best objective: 141.53333333333333
demands = [61, 40, 32, 100]
size, dual = [(2, 0.2), (3, 0.3333333333333333), (5, 0.5), (7, 1.0)]


[([5, 0, 0, 0], 12.2),
 ([0, 3, 0, 0], 13.333333333333334),
 ([0, 0, 2, 0], 16.0),
 ([0, 0, 0, 1], 100.0)]

First creates patterns with only one size per log.

In [3]:
# ----------------------------
# Pricing Problem (Knapsack)
# ----------------------------
print(f"sizes = {sizes}")
new_pattern, reduced_cost = cs_cg.pricing(duals)
print(f"new_pattern = {new_pattern}, reduced_cost = {reduced_cost}")

sizes = [2, 3, 5, 7]
new_pattern = [0, 1, 0, 1], reduced_cost = -0.33333333333333326


In [4]:
patterns.append(new_pattern)
patterns

[[5, 0, 0, 0], [0, 3, 0, 0], [0, 0, 2, 0], [0, 0, 0, 1], [0, 1, 0, 1]]

In [5]:
solver, patterns, q_pattern, duals = cs_cg.master(patterns=patterns)
print("Best objective:", solver.Objective().Value())
print(f"demands = {demands}")
print(f"duals={duals}")
print(f"size, dual = {[(s, d) for s, d in zip(sizes, duals, strict=False)]}")
[(p, q_pattern[i].solution_value()) for i, p in enumerate(patterns)]

Best objective: 128.2
demands = [61, 40, 32, 100]
duals=[0.2, 0.0, 0.5, 1.0]
size, dual = [(2, 0.2), (3, 0.0), (5, 0.5), (7, 1.0)]


[([5, 0, 0, 0], 12.2),
 ([0, 3, 0, 0], 0.0),
 ([0, 0, 2, 0], 16.0),
 ([0, 0, 0, 1], 60.00000000000001),
 ([0, 1, 0, 1], 39.99999999999999)]

In [6]:
cs_cg = CuttingStockColGen(sizes, demands, wood_length=W)
solution, objective = cs_cg.solve(MAX_ITER=10)


 Best objective iteration = 0: 141.53333333333333
demands = [61, 40, 32, 100]
[([5, 0, 0, 0], 12.2), ([0, 3, 0, 0], 13.333333333333334), ([0, 0, 2, 0], 16.0), ([0, 0, 0, 1], 100.0)]
sizes=[2, 3, 5, 7]
new_pattern = [0, 1, 0, 1], reduced_cost = -0.33333333333333326

 Best objective iteration = 1: 128.2
demands = [61, 40, 32, 100]
[([5, 0, 0, 0], 12.2), ([0, 3, 0, 0], 0.0), ([0, 0, 2, 0], 16.0), ([0, 0, 0, 1], 60.00000000000001), ([0, 1, 0, 1], 39.99999999999999)]
sizes=[2, 3, 5, 7]
new_pattern = [1, 0, 0, 1], reduced_cost = -0.19999999999999996

 Best objective iteration = 2: 116.20000000000002
demands = [61, 40, 32, 100]
[([5, 0, 0, 0], 0.19999999999999862), ([0, 3, 0, 0], 0.0), ([0, 0, 2, 0], 16.0), ([0, 0, 0, 1], 0.0), ([0, 1, 0, 1], 39.99999999999999), ([1, 0, 0, 1], 60.000000000000014)]
sizes=[2, 3, 5, 7]
new_pattern = [0, 0, 2, 0], reduced_cost = 0.0
Optimal LP reached.


Solve MIP with chosen patterns

 Objective = 3: 117.0


In [7]:
from cutting_stock_mip_all import CuttingStockMip, initialize_patterns

In [8]:
patterns = initialize_patterns(sizes, W)
print(patterns)
cm = CuttingStockMip(sizes, demands, wood_length=W)
sol = cm.solve(patterns)
print([s for s in sol[0] if s[1] > 0])

[(0, 0, 0, 0), (0, 0, 0, 1), (0, 0, 1, 0), (0, 0, 2, 0), (0, 1, 0, 0), (0, 1, 0, 1), (0, 1, 1, 0), (0, 2, 0, 0), (0, 3, 0, 0), (1, 0, 0, 0), (1, 0, 0, 1), (1, 0, 1, 0), (1, 1, 0, 0), (1, 1, 1, 0), (1, 2, 0, 0), (2, 0, 0, 0), (2, 0, 1, 0), (2, 1, 0, 0), (2, 2, 0, 0), (3, 0, 0, 0), (3, 1, 0, 0), (4, 0, 0, 0), (5, 0, 0, 0)]

 Objective = 3: 117.0
[((0, 0, 2, 0), 16), ((0, 1, 0, 1), 40), ((1, 0, 0, 0), 1), ((1, 0, 0, 1), 60)]
